In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.market_loader import MarketLoader
from src.data.synthetic_sofr_builder import build_term_sofr_curve

from src.term_structure.bootstrapping import (
    bootstrap_dfs_from_sofr,
    bootstrap_dfs_from_treasury,
    build_coupon_structure
)
from src.term_structure.curve_merger import merge_curves
from src.term_structure.curve_interpolator import log_linear_curve_interpolator
from src.term_structure.curve_builder import build_zero_curve

from src.portfolio.swap_object import IRSwap
from src.portfolio.swap_portfolio import SwapPortfolio


In [2]:
### Curve Construction
# downloading market curves
loader = MarketLoader()
curves = loader.loader_pipeline()

# building synthetic sofr curve from ON rates and futures
sofr_curve = build_term_sofr_curve(curves = curves)

# bootstrapping short-end discount factors from sofr curve
df_sofr = bootstrap_dfs_from_sofr(
    sofr_curve = sofr_curve
)

# extracting treasury curve 
treasury_curve = curves['treasury']

# bootstrapping discount factors from treasury curve
df_treasury = bootstrap_dfs_from_treasury(
    treasury_curve = treasury_curve,
    short_dfs = df_sofr
)

# concatenating short-end and long-end into a single curve -> full curve
df_full_curve = merge_curves(
    short_curve = df_sofr,
    long_curve = df_treasury
)

# constructing semi-annual coupon structure
coupon_structure = build_coupon_structure(
    max_year = 30,
    freq = 2
)

# log-linear discount curve interpolation
df_loglinear_interp = log_linear_curve_interpolator(
    df_curve = df_full_curve,
    _target_times = coupon_structure
)

# building zero curve from log-linear discount curve
zero_curve_full = build_zero_curve(
    df_curve = df_loglinear_interp
)

zero_curve_full.tail()


treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..


,0.5,1.0,1.5,2.0,2.5,3.0,3.5,4.0,4.5,5.0,...,25.5,26.0,26.5,27.0,27.5,28.0,28.5,29.0,29.5,30.0
2026-04-15,0.035680,0.036671,0.037068,0.037267,0.037740,0.038056,0.038281,0.038450,0.038582,0.038687,...,0.049899,0.049986,0.050069,0.050149,0.050226,0.050300,0.050372,0.050441,0.050508,0.050573
2026-04-16,0.035483,0.036574,0.037171,0.037469,0.037907,0.038199,0.038408,0.038565,0.038686,0.038784,...,0.050348,0.050436,0.050521,0.050602,0.050681,0.050756,0.050830,0.050900,0.050968,0.051034
2026-04-17,0.035287,0.036080,0.036544,0.036777,0.037216,0.037508,0.037717,0.037874,0.037996,0.038094,...,0.049861,0.049950,0.050036,0.050118,0.050198,0.050275,0.050349,0.050420,0.050489,0.050556
2026-04-20,0.035582,0.036176,0.036641,0.036874,0.037347,0.037663,0.037888,0.038057,0.038189,0.038294,...,0.049850,0.049939,0.050025,0.050108,0.050188,0.050264,0.050339,0.050410,0.050480,0.050546
2026-04-21,0.035680,0.036572,0.037169,0.037468,0.037907,0.038199,0.038407,0.038564,0.038686,0.038783,...,0.049875,0.049960,0.050041,0.050120,0.050196,0.050269,0.050339,0.050407,0.050473,0.050537


In [3]:
### Swap Objects
# creating IR swaps
swap1 = IRSwap(
    maturity = 5,
    fixed_rate = 0.027,
    notional = 1_000_000,
    pay_receive = 'payer',
    freq = 2
)
swap2 = IRSwap(
    maturity = 7,
    fixed_rate = 0.028,
    notional = 2_000_000,
    pay_receive = 'receiver',
    freq = 2
)
swap3 = IRSwap(
    maturity = 10,
    fixed_rate = 0.032,
    notional = 1_500_000,
    pay_receive = 'payer',
    freq = 2
)

swap_portfolio = SwapPortfolio([swap1, swap2, swap3])
swap_portfolio.summary()

,TradeID,Type,Maturity,FixedRate,Notional,CouponFrequency
0,0,payer,5,0.027,1000000,2
1,1,receiver,7,0.028,2000000,2
2,2,payer,10,0.032,1500000,2


In [4]:
# pricing trades in the portfolio
swap_portfolio.price_trades(df_curve = df_loglinear_interp)

,Trade_0,Trade_1,Trade_2
2018-04-03,-4676.730317,-11616.876545,-53665.667446
2018-04-04,-4208.496176,-11072.108559,-53656.640727
2018-04-05,-2803.766877,-6513.524803,-48339.637567
2018-04-06,-5614.722053,-14174.243512,-56337.862054
2018-04-09,-4676.415832,-12347.963926,-54987.500476
...,...,...,...
2026-04-15,54115.600235,160605.128193,133041.546290
2026-04-16,54550.401991,163135.927249,136563.512253
2026-04-17,51486.466746,155717.077591,129648.137119
2026-04-20,52366.665019,156663.257388,129595.929596


In [5]:
# portfolio NPV
swap_portfolio.portfolio_npv(df_curve = df_loglinear_interp)

,Portfolio_NPV
2018-04-03,-69959.274308
2018-04-04,-68937.245462
2018-04-05,-57656.929247
2018-04-06,-76126.827620
2018-04-09,-72011.880233
...,...
2026-04-15,347762.274718
2026-04-16,354249.841492
2026-04-17,336851.681456
2026-04-20,338625.852003


In [6]:
# trade-level DV01 - parallel shock
swap_portfolio.trade_dv01(
    df_curve = df_loglinear_interp,
    shock_bps = 1.0,
    shock_type = 'parallel'
)

,Trade_0,Trade_1,Trade_2
2018-04-03,473.1,1287.7,1344.1
2018-04-04,472.9,1287.4,1344.1
2018-04-05,472.2,1284.2,1339.0
2018-04-06,473.6,1289.5,1346.6
2018-04-09,473.1,1288.3,1345.4
...,...,...,...
2026-04-15,444.5,1171.2,1166.9
2026-04-16,444.2,1169.4,1163.4
2026-04-17,445.7,1174.4,1169.9
2026-04-20,445.3,1173.8,1170.1


In [7]:
# portfolio DV01 - parallel shock
swap_portfolio.portfolio_dv01(
    df_curve = df_loglinear_interp,
    shock_bps = 1.0,
    shock_type = 'parallel'
)

,Portfolio_DV01
2018-04-03,3104.9
2018-04-04,3104.4
2018-04-05,3095.4
2018-04-06,3109.7
2018-04-09,3106.8
...,...
2026-04-15,2782.6
2026-04-16,2777.0
2026-04-17,2790.0
2026-04-20,2789.2


In [8]:
# trade-level DV01 - key-rate
key_rates = [2.0, 5.0, 10.0]

swap_portfolio.trade_dv01(
    df_curve = df_loglinear_interp,
    shock_bps = 1.0,
    shock_type = 'key_rate',
    key_rate_tenors = key_rates
)

Trade_0             Trade_1            Trade_2              
              2.0    5.0  10.0    2.0   5.0  10.0    2.0   5.0     10.0
2018-04-03     2.6  445.0  0.0     5.4  12.3  0.0     4.6  10.5  1152.4
2018-04-04     2.6  444.8  0.0     5.4  12.3  0.0     4.6  10.5  1152.5
2018-04-05     2.6  444.1  0.0     5.3  12.3  0.0     4.6  10.5  1147.8
2018-04-06     2.6  445.5  0.0     5.4  12.3  0.0     4.6  10.5  1154.7
2018-04-09     2.6  445.0  0.0     5.3  12.3  0.0     4.6  10.5  1153.6
...            ...    ...  ...     ...   ...  ...     ...   ...     ...
2026-04-15     2.5  417.5  0.0     5.2  11.5  0.0     4.5   9.9   991.5
2026-04-16     2.5  417.3  0.0     5.2  11.5  0.0     4.5   9.9   988.3
2026-04-17     2.5  418.8  0.0     5.2  11.6  0.0     4.5   9.9   994.1
2026-04-20     2.5  418.3  0.0     5.2  11.6  0.0     4.5   9.9   994.3
2026-04-21     2.5  417.3  0.0     5.2  11.5  0.0     4.5   9.9   990.5

[2101 rows x 9 columns]

In [9]:
# trade-level DV01 - multi_tenor
steepener = {
    2.0: -1.0,
    5.0: -0.5,
    10.0: +1.0,
    30.0: +1.5 
}

swap_portfolio.trade_dv01(
    df_curve = df_loglinear_interp,
    shock_type = 'multi_tenor',
    multi_tenor_dict = steepener
)

,Trade_0,Trade_1,Trade_2
2018-04-03,-225.2,-11.5,1142.5
2018-04-04,-225.1,-11.5,1142.6
2018-04-05,-224.7,-11.5,1137.9
2018-04-06,-225.4,-11.5,1144.8
2018-04-09,-225.2,-11.5,1143.8
...,...,...,...
2026-04-15,-211.3,-11.0,982.1
2026-04-16,-211.2,-11.0,978.9
2026-04-17,-212.0,-11.0,984.7
2026-04-20,-211.8,-11.0,984.9
